# Ordered Logistic Regression Results (FAIR²) Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll demonstrate how to inspect metadata, discover record sets, extract tabular data using `@id` references, and conduct exploratory data analysis.

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant, if not already installed
!pip install -U mlcroissant

## 1. Data Loading

Here we load the dataset metadata and prepare for record exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load metadata and instantiate the croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
md = dataset.metadata
print(f"{md.name}\n---\n{md.description}")

## 2. Data Overview

List record sets (`RecordSet`), their fields, and corresponding `@id`. All exploration should be referenced by `@id`, as per FAIR and Croissant best practices.

In [ ]:
# List available record sets and their @id
from pprint import pprint

# The available record sets are accessible via dataset.metadata.recordSets
record_sets = dataset.metadata.recordSets

if not record_sets:
    print("No RecordSet objects found in the schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs.name} (@id: {rs.id})")
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f.id}, dataType: {getattr(f, 'dataType', None)})")

## 3. Data Extraction

Load records from a record set into a DataFrame for exploration. We'll use precise `@id` references for each RecordSet and field, and store them in variables for clear code. If no RecordSet is found, this step will be skipped.

In [ ]:
# Extract tabular data by RecordSet @id
dataframes = {}
if not record_sets:
    print("No available record sets to load records from.")
else:
    record_set_ids = [rs.id for rs in record_sets]
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id}, shape: {df.shape}")
    # Show columns and head for the first RecordSet
    first_rs = record_set_ids[0]
    print(f"\nColumns for RecordSet @id {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Typical EDA includes filtering, normalization, and grouping of data. We'll use field and record set references by `@id` throughout. Adjust the numeric and grouping fields according to those presented in the overview above.

In [ ]:
# EDA: filter, normalize, group by field @id
if not record_sets:
    print("No data available for EDA as no RecordSet present.")
else:
    # Select first RecordSet as example
    rs = record_sets[0]
    rs_id = rs.id
    df = dataframes[rs_id]

    # Find a numeric candidate field from schema for demonstration
    numeric_fields = [f for f in rs.fields if hasattr(f, 'dataType') and f.dataType in ('schema:Integer','schema:Float','schema:Number')]
    if numeric_fields:
        num_field = numeric_fields[0].id
        print(f"Using numeric field: {num_field}")
        threshold = 10
        if num_field in df.columns and pd.api.types.is_numeric_dtype(df[num_field]):
            filtered_df = df[df[num_field] > threshold]
        else:
            # Try to coerce field to numeric then filter
            filtered_df = df.copy()
            filtered_df[num_field] = pd.to_numeric(filtered_df[num_field], errors='coerce')
            filtered_df = filtered_df[filtered_df[num_field] > threshold]
        print(f"Filtered records with {num_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        col_norm = f"{num_field}_normalized"
        filtered_df[col_norm] = (filtered_df[num_field] - filtered_df[num_field].mean()) / filtered_df[num_field].std()
        print("\nNormalized values:")
        print(filtered_df[[num_field, col_norm]].head())

        # Group by a non-numeric field if available
        group_fields = [f for f in rs.fields if hasattr(f, 'dataType') and f.dataType not in ('schema:Integer','schema:Float','schema:Number')]
        if group_fields:
            group_field = group_fields[0].id
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[num_field].mean()
                print(f"\nGrouped mean {num_field} by {group_field}:")
                print(grouped_df.head())
            else:
                print(f"Group field {group_field} not in filtered DataFrame columns.")
        else:
            print("No suitable non-numeric field for grouping found.")
    else:
        print("No suitable numeric field found for EDA.")

## 5. Visualization
Plot field distributions or relationships using the DataFrame. Adjust the field `@id` references as per data.

In [ ]:
import matplotlib.pyplot as plt

if not record_sets:
    print("No data available for visualization.")
else:
    rs = record_sets[0]
    rs_id = rs.id
    df = dataframes[rs_id]

    numeric_fields = [f for f in rs.fields if hasattr(f, 'dataType') and f.dataType in ('schema:Integer','schema:Float','schema:Number')]
    if numeric_fields:
        num_field = numeric_fields[0].id
        plt.figure(figsize=(7,4))
        pd.to_numeric(df[num_field], errors='coerce').dropna().plot.hist(bins=20, alpha=0.7)
        plt.title(f"Distribution of field: {num_field}")
        plt.xlabel(num_field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric fields found to visualize.")

## 6. Conclusion

In this notebook, you explored the FAIR² dataset using the Croissant schema and `mlcroissant`. We demonstrated how to discover data structure via metadata, referenced all entities by their `@id` per best practices, and performed tabular data analysis using pandas. The precise field and record set references ensure full reproducibility and transparency of your workflow. For advanced workflow, check the [mlcroissant documentation](https://mlcroissant.io/).

_Adjust the analysis steps as needed for your own project or deeper research._